In [1]:
import numpy as np 
import pandas as pd

In [2]:
train = pd.read_csv('../data/processed/train_processed.csv')
test = pd.read_csv('../data/processed/test_processed.csv')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')  

Train shape: (48120, 5)
Test shape: (11808, 4)


In [3]:
print(train.head())

              DateTime  Junction  Vehicles           ID  is_train
0  2015-11-01 00:00:00         1        15  20151101001         0
1  2015-11-01 01:00:00         1        13  20151101011         0
2  2015-11-01 02:00:00         1        10  20151101021         0
3  2015-11-01 03:00:00         1         7  20151101031         0
4  2015-11-01 04:00:00         1         9  20151101041         0


In [4]:
print(test.head())

              DateTime  Junction           ID  is_train
0  2017-07-01 00:00:00         1  20170701001         1
1  2017-07-01 01:00:00         1  20170701011         1
2  2017-07-01 02:00:00         1  20170701021         1
3  2017-07-01 03:00:00         1  20170701031         1
4  2017-07-01 04:00:00         1  20170701041         1


In [5]:
#Combine data for feature engineering
all = pd.concat([train, test], axis=0, ignore_index=True)

In [6]:
print(all.head()) #Check combined data head

              DateTime  Junction  Vehicles           ID  is_train
0  2015-11-01 00:00:00         1      15.0  20151101001         0
1  2015-11-01 01:00:00         1      13.0  20151101011         0
2  2015-11-01 02:00:00         1      10.0  20151101021         0
3  2015-11-01 03:00:00         1       7.0  20151101031         0
4  2015-11-01 04:00:00         1       9.0  20151101041         0


In [7]:
print(all.tail()) #Check combined data tail

                  DateTime  Junction  Vehicles           ID  is_train
59923  2017-10-31 19:00:00         4       NaN  20171031194         1
59924  2017-10-31 20:00:00         4       NaN  20171031204         1
59925  2017-10-31 21:00:00         4       NaN  20171031214         1
59926  2017-10-31 22:00:00         4       NaN  20171031224         1
59927  2017-10-31 23:00:00         4       NaN  20171031234         1


In [8]:
print(all.dtypes) #Check data types

DateTime        str
Junction      int64
Vehicles    float64
ID            int64
is_train      int64
dtype: object


In [9]:
all['DateTime'] = pd.to_datetime(all['DateTime']) #convert DateTime column to datetime format

In [10]:
# Sort by Junction and DateTime for time-based feature engineering and ensure correct order for lag features
all = all.sort_values(by=['Junction', 'DateTime']).reset_index(drop=True)

In [11]:
#Time Features
all['hour'] = all['DateTime'].dt.hour
all['day_of_week'] = all['DateTime'].dt.dayofweek
all['day_of_month'] = all['DateTime'].dt.day
all['month'] = all['DateTime'].dt.month
all['year'] = all['DateTime'].dt.year
all['week_of_year'] = all['DateTime'].dt.isocalendar().week.astype(int) #Convert to int for compatibility
all['quarter'] = all['DateTime'].dt.quarter

In [12]:
#Boolean features for rush hours (7-9 AM and 4-6 PM) and weekends
all['is_weekend'] = (all['day_of_week'] >= 5).astype(int) # 1 for Sunday, 0 for Monday
all['is_rush_hour'] = all['hour'].isin([7, 8, 9, 16, 17, 18, 19]).astype(int) # 1 for rush hours, 0 otherwise
all['is_night'] = (all['hour'].between(22, 23) | all['hour'].between(0, 5)) # 1 for night hours, 0 otherwise

In [13]:
#Cyclical features for hour and day of week
all['hour_sin'] = np.sin(2 * np.pi * all['hour'] / 24)
all['hour_cos'] = np.cos(2 * np.pi * all['hour'] / 24)

all['dow_sin'] = np.sin(2 * np.pi * all['day_of_week'] / 7)
all['dow_cos'] = np.cos(2 * np.pi * all['day_of_week'] / 7)

all['month_sin'] = np.sin(2 * np.pi * all['month'] / 12)
all['month_cos'] = np.cos(2 * np.pi * all['month'] / 12)


In [15]:
#Lag features 
# Process each junction separately to avoid data leakage between junctions
grouped = all.groupby('Junction')['Vehicles'] #Group by Junction for lag features

In [16]:
#Lag features for traffic volume (previous hour, previous day same hour, previous week same hour)
all['lag_1h'] = grouped.shift(1)
all['lag_2h'] = grouped.shift(2)
all['lag_24h'] = grouped.shift(24)
all['lag_168h'] = grouped.shift(168)


In [17]:
#Rolling statistics (mean and std) for previous 3, 6, 12, 24 hours

all['rolling_mean_3h'] = grouped.transform(lambda x: x.rolling(3).mean())
all['rolling_mean_24h'] = grouped.transform(lambda x: x.rolling(24).mean())
all['rolling_std_24h'] = grouped.transform(lambda x: x.rolling(24).std())
all['rolling_mean_7d'] = grouped.transform(lambda x: x.rolling(168).mean())

In [21]:
#Split back into train and test
train_features = all[all['is_train'] == 0].copy()
train_features = train_features.drop(columns=['is_train'])

test_features = all[all['is_train'] == 1].copy()
test_features = test_features.drop(columns=['is_train', 'Vehicles'])

In [22]:
print(train_features.head())
print(test_features.head())

             DateTime  Junction  Vehicles           ID  hour  day_of_week  \
0 2015-11-01 00:00:00         1      15.0  20151101001     0            6   
1 2015-11-01 01:00:00         1      13.0  20151101011     1            6   
2 2015-11-01 02:00:00         1      10.0  20151101021     2            6   
3 2015-11-01 03:00:00         1       7.0  20151101031     3            6   
4 2015-11-01 04:00:00         1       9.0  20151101041     4            6   

   day_of_month  month  year  week_of_year  ...  month_sin  month_cos  lag_1h  \
0             1     11  2015            44  ...       -0.5   0.866025     NaN   
1             1     11  2015            44  ...       -0.5   0.866025    15.0   
2             1     11  2015            44  ...       -0.5   0.866025    13.0   
3             1     11  2015            44  ...       -0.5   0.866025    10.0   
4             1     11  2015            44  ...       -0.5   0.866025     7.0   

   lag_2h  lag_24h  lag_168h  rolling_mean_3h  rol

In [23]:
train_features.to_csv('../data/features/train_features.csv', index=False)
test_features.to_csv('../data/features/test_features.csv', index=False)

In [24]:
print(f"\n✅ Feature Engineering Complete!")
print(f"Final Train Shape: {train_features.shape} (Ready for 80/20 split)")
print(f"Final Test Shape:  {test_features.shape} (Blind data)")


✅ Feature Engineering Complete!
Final Train Shape: (48120, 28) (Ready for 80/20 split)
Final Test Shape:  (11808, 27) (Blind data)
